# QuantConnect MES/MNQ individual-expiry minute-history feasibility

**Purpose:** Verify whether the QuantConnect Research environment can enumerate historical MES/MNQ expiry contracts and request their individual one-minute TradeBar history on the current account.

**Data policy:** All OHLCV remains in QuantConnect memory. The notebook displays only aggregate contract counts, bar counts, active-date counts and gap counts. Do not print, chart, save, download, or transmit raw market-data values or rows. This is not an attempt to bypass QuantConnect data licensing.

**Scope:** 4 Nov 2025 through 3 Nov 2026 (end-exclusive 4 Nov 2026). This is a feasibility test, not the final frozen strategy evaluation.

## Run instructions

1. Open QuantConnect Cloud and create/open a **Research Notebook** (not a local Jupyter notebook).
2. Copy the code cell below into the notebook and run it.
3. It requests the historical FutureUniverse chain for MES and MNQ, then requests minute TradeBar history separately for each returned expiry contract.
4. Only aggregate summary tables are displayed. Do not add code that exports or prints bars, prices, or contract-level data values.
5. If a history request errors or is denied by plan entitlements, copy the exact error text. Do not change to paid access.

In [ ]:
from AlgorithmImports import *
from datetime import datetime, timedelta
from collections import defaultdict
import pandas as pd

# Free-plan feasibility probe. Keep all raw bars inside QuantConnect.
# Only aggregate counts/coverage are displayed. No orders, file writes, or exports.

START = datetime(2025, 11, 4)
END = datetime(2026, 11, 4)  # exclusive
ROOTS = {
    "MES": Futures.Indices.MicroSP500EMini,
    "MNQ": Futures.Indices.MicroNASDAQ100EMini,
}

qb = QuantBook()
root_results = []
contract_results = []

for root, future_ticker in ROOTS.items():
    future = qb.add_future(
        future_ticker,
        Resolution.MINUTE,
        data_mapping_mode=DataMappingMode.OPEN_INTEREST,
        data_normalization_mode=DataNormalizationMode.RAW,
        contract_depth_offset=0,
    )

    chain = qb.history(
        FutureUniverse,
        future.symbol,
        START,
        END,
        flatten=True,
    )

    if chain is None or chain.empty:
        root_results.append({
            "root": root,
            "chain_rows": 0,
            "contracts_seen": 0,
            "contracts_with_minute_bars": 0,
            "individual_contract_minute_bars": 0,
            "active_contract_sessions": 0,
            "gap_intervals_over_1m": 0,
            "status": "NO_FUTURE_UNIVERSE_HISTORY",
        })
        continue

    if "symbol" in chain.index.names:
        contract_symbols = list(chain.index.get_level_values("symbol").unique())
    elif "symbol" in chain.columns:
        contract_symbols = list(chain["symbol"].dropna().unique())
    else:
        raise RuntimeError(
            "Unexpected FutureUniverse history shape. Inspect schema only; "
            "do not print market-data rows."
        )

    total_bars = 0
    active_sessions = set()
    contracts_with_bars = 0
    gap_intervals = 0
    contract_counts = []

    for contract_symbol in contract_symbols:
        bars = qb.history(
            TradeBar,
            contract_symbol,
            START,
            END,
            Resolution.MINUTE,
        )
        if bars is None or bars.empty:
            contract_counts.append(0)
            continue

        # Work with timestamps only. Never display or export OHLCV values.
        time_level = "time" if "time" in bars.index.names else bars.index.names[-1]
        times = pd.DatetimeIndex(bars.index.get_level_values(time_level)).sort_values()
        unique_times = times.unique()
        unique_times = unique_times.sort_values()
        contract_bar_count = len(unique_times)
        contract_counts.append(contract_bar_count)
        if contract_bar_count:
            contracts_with_bars += 1
        total_bars += contract_bar_count

        for timestamp in unique_times:
            active_sessions.add(timestamp.date())

        for previous, current in zip(unique_times[:-1], unique_times[1:]):
            if (current - previous).total_seconds() > 60:
                gap_intervals += 1

        contract_results.append({
            "root": root,
            "minute_bars": contract_bar_count,
            "active_dates": len({timestamp.date() for timestamp in unique_times}),
            "first_bar_date": str(unique_times[0].date()) if contract_bar_count else None,
            "last_bar_date": str(unique_times[-1].date()) if contract_bar_count else None,
        })

    root_results.append({
        "root": root,
        "chain_rows": int(len(chain)),
        "contracts_seen": int(len(contract_symbols)),
        "contracts_with_minute_bars": int(contracts_with_bars),
        "individual_contract_minute_bars": int(total_bars),
        "active_contract_sessions": int(len(active_sessions)),
        "gap_intervals_over_1m": int(gap_intervals),
        "max_bars_in_one_contract": int(max(contract_counts, default=0)),
        "status": "INDIVIDUAL_CONTRACT_HISTORY_RECEIVED" if total_bars else "NO_INDIVIDUAL_CONTRACT_BARS",
    })

# Aggregate-only display. No prices, raw bars, contract IDs, or market-data rows.
root_summary = pd.DataFrame(root_results)
contract_summary = pd.DataFrame(contract_results)

display(root_summary)
display(
    contract_summary.groupby("root").agg(
        contracts_with_history=("minute_bars", "count"),
        median_bars_per_contract=("minute_bars", "median"),
        max_bars_per_contract=("minute_bars", "max"),
        total_active_dates=("active_dates", "sum"),
    ).reset_index()
    if not contract_summary.empty
    else pd.DataFrame(columns=["root", "contracts_with_history", "median_bars_per_contract", "max_bars_per_contract", "total_active_dates"])
)


## Interpretation

- `INDIVIDUAL_CONTRACT_HISTORY_RECEIVED` means the Research environment returned individual contract minute history for that root in the tested date window.
- A positive chain count with zero individual bars means contract enumeration worked but individual history did not.
- The aggregate gap count is only a first diagnostic. Overnight closures, maintenance windows, weekends and holidays naturally create gaps; it is not a missing-data verdict.
- The notebook does not yet run the ORB15/VWAP/compression strategies and does not establish profitability.
- A successful notebook run does not prove the full 90-day holdout or free-tier compute limits are sufficient. The next step is a controlled in-cloud evaluator with fixed chronological development/holdout dates and only aggregate permitted outputs.

## Official documentation

- Futures historical data and `FutureUniverse` history: https://www.quantconnect.com/docs/v2/writing-algorithms/historical-data/asset-classes/futures
- Individual futures contracts: https://www.quantconnect.com/docs/v2/writing-algorithms/securities/asset-classes/futures/requesting-data/individual-contracts
- QuantConnect US Futures dataset: https://www.quantconnect.com/docs/v2/writing-algorithms/datasets/algoseek/us-futures

**Important:** The free-plan entitlement and the account's actual available history must be checked in the logged-in QuantConnect environment. Do not infer a successful data entitlement from public documentation alone.